### **Overview of the Transformer Decoder:**
- **Purpose**: Generates output sequences (e.g., translated text, next words) using encoded information and previously generated tokens.
- **Architecture**:
  - **Masked Self-Attention**:
    - Focuses on already generated tokens in the output sequence.
    - Masks future tokens to prevent "cheating" (ensures autoregressive generation).
  - **Encoder-Decoder Attention**:
    - Attends to the encoded input (from the encoder) to integrate context into the output sequence.
  - **Feedforward Layers**:
    - Processes the combined output of attention mechanisms through fully connected layers for transformation.
- **Token Prediction**:
  - Produces a probability distribution over the vocabulary for the next token using a softmax layer.
- **Generation Process**:
  - Decodes one token at a time autoregressively until the end of the sequence.

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load pre-trained GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Input prompt for the decoder
prompt = "The cat sat on the mat and"

# Encode the prompt to tokens
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Generate text using the decoder
output = model.generate(
    input_ids,
    max_length=50,  # Maximum number of tokens to generate
    num_return_sequences=1,  # Number of output sequences
    no_repeat_ngram_size=2,  # Avoid repeating phrases
    temperature=0.7,  # Sampling temperature (controls randomness)
    top_k=50,  # Limits the sampling to top-k tokens
    top_p=0.95,  # Nucleus sampling (top-p tokens)
)

# Decode the output tokens to text
decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)

# Print the generated text
print("Generated Text:\n", decoded_output)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpec

Generated Text:
 The cat sat on the mat and looked at the ceiling.

"I'm not sure what you're talking about," she said. "I don't know what to say."
. . .
,
I've been thinking about this


### **Overview of the HDC Decoder**:
- **Purpose**: Recovers or interprets encoded information (hypervectors) by comparing them to a reference set of known hypervectors.
- **Key Steps**:
  - **Similarity Matching**:
    - Compares the encoded hypervector to pre-stored hypervectors (e.g., using cosine similarity or dot product).
  - **Probabilistic Interpretation**:
    - Selects the closest match based on the highest similarity score.
  - **Approximation**:
    - Decoding is approximate and retrieves the "most similar" representation rather than reconstructing the exact original input.
- **Applications**:
  - Used in tasks like classification, pattern recognition, and associative memory retrieval.

In [ ]:
import numpy as np

# Define high-dimensional space (e.g., 10,000 dimensions)
dimension = 10000

# Initialize random seed for reproducibility
np.random.seed(42)

# Example tokens and their hypervectors
unique_tokens = ['cat', 'mat', 'sat', 'on', 'the']
hypervectors = {token: np.random.choice([-1, 1], size=dimension) for token in unique_tokens}

# Example encoded sentence hypervector (bundling)
encoded_sentence = np.sign(
    np.sum([hypervectors['the'], hypervectors['cat'], hypervectors['sat'], hypervectors['on'], hypervectors['the'], hypervectors['mat']], axis=0)
)

# Reference set of hypervectors (e.g., token hypervectors)
reference_hypervectors = hypervectors

# HDC Decoder: Compare encoded sentence to reference hypervectors
def hdc_decoder(encoded_vector, reference_hypervectors):
    similarities = {}
    for token, ref_vector in reference_hypervectors.items():
        # Cosine similarity
        similarity = np.dot(encoded_vector, ref_vector) / (np.linalg.norm(encoded_vector) * np.linalg.norm(ref_vector))
        similarities[token] = similarity
    return similarities

# Decode the encoded sentence
decoded_similarities = hdc_decoder(encoded_sentence, reference_hypervectors)

# Output the most similar tokens
most_similar_tokens = sorted(decoded_similarities.items(), key=lambda x: x[1], reverse=True)
most_similar_tokens

[('the', 0.7302216587121048),
 ('sat', 0.30027158425673445),
 ('mat', 0.29611077708458566),
 ('cat', 0.2852464472461973),
 ('on', 0.2699901542816519)]

Transformers Encoding and Decoding Process (Short Overview)
Encoding:

Input Tokenization: The input text is split into tokens and converted into numeric IDs.
Embedding: Tokens are mapped to dense vectors via learned embeddings.
Positional Encoding: Adds positional information to the embeddings for sequence awareness.
Self-Attention: Calculates relationships between all tokens using attention scores, generating context-aware representations.
Feedforward Layers: Refines token representations layer by layer.
Decoding:

Input for Decoder: Takes the encoder outputs and previously generated tokens.
Masked Self-Attention: Computes relationships among already generated tokens while masking future tokens to maintain autoregressive behavior.
Encoder-Decoder Attention: Incorporates context from encoder outputs into the generated sequence.
Feedforward Layers: Refines and transforms token predictions.
Output Generation: Produces logits for the next token, selects a token (e.g., via softmax), and repeats until sequence completion.
Key Characteristics:

Encoding is context-aware and probabilistic, driven by attention mechanisms.
Decoding is deterministic in generating outputs (though strategies like beam search or sampling may add variability).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Step 1: Load a pre-trained Transformer model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-small")  # Using T5 as an example
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Step 2: Input text for encoding and decoding
input_text = "The cat sat on the mat."

# Step 3: Tokenization (Embedding Step)
input_ids = tokenizer.encode(input_text, return_tensors="pt")  # Convert text to token IDs

# Step 4: Encoding (Contextualized Representations)
encoder_outputs = model.encoder(input_ids)  # Pass input through the encoder
encoder_hidden_states = encoder_outputs.last_hidden_state  # Contextual embeddings

# Step 5: Decoding (Generating Text)
# Provide the encoded context to the decoder with a prefix (e.g., translating, summarizing, etc.)
decoder_input_ids = tokenizer.encode("Summarize:", return_tensors="pt")  # Prompt the decoder
output_ids = model.generate(
    input_ids=input_ids,  # Encoded input context
    max_length=20,        # Limit the length of the output
    num_beams=5,          # Use beam search for more diverse outputs
    early_stopping=True   # Stop decoding early if EOS token is reached
)

# Step 6: Decoded Output (Final Text Generation)
decoded_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Display Results
{
    "Input Text": input_text,
    "Tokenized Input IDs": input_ids.tolist(),
    "Encoder Hidden States Shape": encoder_hidden_states.shape,
    "Generated Output Text": decoded_output,
}

HDC Encoding and Decoding Process (Short Overview)
Encoding:

Token Representation: Each unique token is assigned a high-dimensional binary vector (hypervector) randomly initialized to ensure near-orthogonality.
Binding: Relationships between features or tokens are encoded using operations like XOR or element-wise multiplication.
Bundling: Multiple token hypervectors are aggregated (e.g., summed) into a single hypervector to represent the whole input.
Binarization: The aggregated vector is converted back to a binary hypervector using the sign function (values become -1 or +1).
Decoding:

Similarity Matching: The encoded hypervector is compared against a reference set of hypervectors using similarity measures like cosine similarity or dot product.
Probabilistic Interpretation: The closest match (highest similarity) is selected as the decoded result.
Approximation: Decoding retrieves patterns or categories rather than reconstructing the exact input.
Key Characteristics:

Encoding is deterministic, relying on mathematical operations like summation and binarization.
Decoding is probabilistic, retrieving the "most similar" representation based on the encoded hypervector.

In [ ]:
import numpy as np

# Define high-dimensional space (e.g., 10,000 dimensions)
dimension = 10000

# Initialize random seed for reproducibility
np.random.seed(42)

# Step 1: Token Representation (Embedding)
tokens = ["the", "cat", "sat", "on", "the", "mat"]
unique_tokens = set(tokens)
hypervectors = {token: np.random.choice([-1, 1], size=dimension) for token in unique_tokens}

# Step 2: Encoding (Aggregate Hypervectors)
# Binding is skipped as this example focuses on bundling
encoded_sentence = np.sign(np.sum([hypervectors[token] for token in tokens], axis=0))

# Step 3: Decoding (Similarity Matching)
def hdc_decoder(encoded_vector, reference_hypervectors):
    similarities = {}
    for token, ref_vector in reference_hypervectors.items():
        # Cosine similarity
        similarity = np.dot(encoded_vector, ref_vector) / (np.linalg.norm(encoded_vector) * np.linalg.norm(ref_vector))
        similarities[token] = similarity
    return similarities

# Decode the sentence by comparing to reference hypervectors
decoded_similarities = hdc_decoder(encoded_sentence, hypervectors)
most_similar_tokens = sorted(decoded_similarities.items(), key=lambda x: x[1], reverse=True)

# Display Results
{
    "Input Tokens": tokens,
    "Hypervector Encoding": encoded_sentence[:10],  # Show the first 10 dimensions for brevity
    "Decoded Similarities": most_similar_tokens,
}

{'Input Tokens': ['the', 'cat', 'sat', 'on', 'the', 'mat'],
 'Hypervector Encoding': array([-1,  1,  0,  1, -1,  1,  1, -1, -1, -1]),
 'Decoded Similarities': [('the', 0.7233506144877193),
  ('mat', 0.30378652180349036),
  ('cat', 0.2957224123889115),
  ('on', 0.286045481091417),
  ('sat', 0.27245169664969837)]}